# Aura Fizz asset generator (free)

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Then run the cells **one by one, top to bottom** (click ▶ on each). The first run downloads the model (about 5 minutes).

Every image uses `coffee_front_final.png` as the style reference.

In [ ]:
# 1. Install (about 2 minutes)
!pip -q install -U git+https://github.com/huggingface/diffusers.git transformers accelerate sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Load the model and your style reference (about 5 minutes the first time)
import torch, json, requests, io, os
from PIL import Image
from diffusers import Flux2KleinPipeline

RAW = "https://raw.githubusercontent.com/HazeEve/movie-aniyomi/claude/gallant-lovelace-tp39bm/AuraFizz/art"
ref = Image.open(io.BytesIO(requests.get(RAW + "/generated/coffee_front_final.png").content)).convert("RGB")
batches = requests.get(RAW + "/tripo/batches.json").json()["batches"]
print(len(batches), "batches loaded")

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
pipe = Flux2KleinPipeline.from_pretrained("black-forest-labs/FLUX.2-klein-4B", torch_dtype=dtype)
pipe.enable_model_cpu_offload()
os.makedirs("/content/out", exist_ok=True)
ref.resize((683, 384))

In [ ]:
# 3. Generate. Change FIRST and LAST to pick which batches (1 to 79).
FIRST, LAST = 1, 3
TRIES = 2          # images per batch, so you can pick the best one

LEAD = ("Use the attached image only as the ART STYLE reference: copy its exact outline weight, cel shading, "
        "warm lamp lighting, colours and cute chunky shapes. Do NOT copy its scene, wall, counter or objects. "
        "Draw a completely new image: ")

from IPython.display import display
for n in range(FIRST, LAST + 1):
    b = batches[n - 1]
    for t in range(TRIES):
        img = pipe(image=[ref], prompt=LEAD + b["prompt"], width=1360, height=768,
                   num_inference_steps=4, guidance_scale=1.0,
                   generator=torch.Generator("cuda").manual_seed(1000 * n + t)).images[0]
        path = f"/content/out/batch{n:02d}_{t+1}.png"
        img.save(path)
        print(path, "-", b["title"])
        display(img.resize((680, 384)))

In [ ]:
# 4. Download everything you made as one zip
!cd /content && zip -qr aurafizz_batches.zip out
from google.colab import files
files.download("/content/aurafizz_batches.zip")

Send me the zip (or the best image of each batch). I cut every object out, name it, and colour-match it to the coffee scene.

If you see **CUDA out of memory**: Runtime → Restart session, then run cells 2 and 3 again.
Free Colab gives a few hours of GPU per day; when it stops, continue tomorrow from the next batch number.